# Support Vector Machines (SVM) for Agricultural Classification

This tutorial notebook demonstrates the complete machine learning workflow using **Support Vector Machines (SVM)** to classify agricultural crop stress and health conditions from phenomic and spectral indicators.

### Table of Contents:
1. **Exploratory Data Analysis** of Crop Phenomics (NDVI, NDRE, Canopy Temp, Chlorophyll, Leaf Moisture)
2. **Feature Standardization** (Essential for margin optimization in SVM)
3. **Kernel Formulation & Comparison** (Linear vs. RBF vs. Polynomial)
4. **Support Vector Analysis** (Inspecting margin boundary vectors)
5. **Hyperparameter Tuning** ($C$ and $\gamma$ grid search)
6. **Interactive Visualizations** with Plotly


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Load crop disease & phenomics dataset
df = pd.read_csv('../data/crop_disease_phenomics.csv')
df.head(10)


## 1. Feature Preprocessing & Scaling

SVM seeks the maximum margin hyperplane based on Euclidean distance ($||w||$). Features with larger scales would dominate the margin without standardization.


In [ ]:
features = ['ndvi', 'ndre', 'canopy_temp_c', 'chlorophyll_spad', 'leaf_moisture_pct']
target = 'condition'

X = df[features].values
y = df[target].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print('Scaled training shape:', X_train_scaled.shape)


## 2. Comparing SVM Kernels with Cross-Validation

We test Linear, Polynomial, and Radial Basis Function (RBF) kernels.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for k in ['linear', 'poly', 'rbf']:
    clf = SVC(kernel=k, random_state=42)
    scores = cross_val_score(clf, X_train_scaled, y_train, cv=cv, scoring='accuracy')
    print(f'Kernel: {k:<8} | 5-Fold Accuracy: {scores.mean():.4f} (+/- {scores.std():.4f})')


## 3. RBF SVM Hyperparameter Optimization (Grid Search)

Optimize $C$ (regularization strength) and $\gamma$ (RBF kernel influence radius).


In [ ]:
param_grid = {
    'C': [0.1, 1.0, 5.0, 20.0],
    'gamma': [0.01, 0.1, 0.5, 1.0]
}
grid = GridSearchCV(SVC(kernel='rbf'), param_grid, cv=5, scoring='accuracy')
grid.fit(X_train_scaled, y_train)

print('Best Hyperparameters:', grid.best_params_)
print(f'Best Cross-Validation Score: {grid.best_score_ * 100:.2f}%')


## 4. Test Set Evaluation & Support Vectors


In [ ]:
best_svm = grid.best_estimator_
preds = best_svm.predict(X_test_scaled)

print('Test Set Accuracy:', accuracy_score(y_test, preds))
print(classification_report(y_test, preds))

print('Total Support Vectors:', len(best_svm.support_))
for cls_name, count in zip(best_svm.classes_, best_svm.n_support_):
    print(f'  - Class {cls_name}: {count} support vectors')


## 5. Interactive 3D Phenomics Space (Plotly)


In [ ]:
fig = px.scatter_3d(
    df, x='ndvi', y='canopy_temp_c', z='chlorophyll_spad',
    color='condition',
    title='Crop Stress Clusters in Phenomics Space',
    labels={'ndvi': 'NDVI', 'canopy_temp_c': 'Canopy Temp (°C)', 'chlorophyll_spad': 'SPAD'},
    color_discrete_map={'Healthy': '#059669', 'Nutrient_Deficiency': '#d97706', 'Fungal_Blight': '#dc2626'}
)
fig.update_layout(template='plotly_white')
fig.show()
